In [4]:
import numpy as np
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_body_barycentric_posvel, solar_system_ephemeris 
import astropy.units as u

In [13]:
%%timeit

a = []
for x in range(0, 100000):
    a.append(x)

148 ns ± 0.332 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [14]:
%%timeit

a = np.arange(100000)

172 ns ± 3.6 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [15]:
%%timeit

arrA = [ x for x in range(0, 100000) ]
arrB = [ x for x in range(0, 100000) ]

result = []
for a, b in zip(arrA, arrB):
    result.append(a + b)

5.04 ms ± 23.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
%%timeit

arrA = np.arange(0, 100000)
arrB = np.arange(0, 100000)

result = arrA + arrB

67.7 µs ± 48.5 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [30]:
from numpy import sin, cos
import numpy as np

def Sky_basis(ra_rad, dec_rad):
        l_hat = np.array([cos(dec_rad)*cos(ra_rad), # vector that points to object's pos on sky from observer
                          cos(dec_rad)*sin(ra_rad),
                          sin(dec_rad)])
        e_ra  = np.array([-sin(ra_rad),  cos(ra_rad), 0.0]) # vector towards increasing RA - points east, 90 deg to l_hat
        e_dec = np.array([-sin(dec_rad)*cos(ra_rad), # vector towards increasing dec - points north
                          -sin(dec_rad)*sin(ra_rad),
                          cos(dec_rad)])
        return l_hat, e_ra, e_dec # turning angular rates into linear velocities

AU_km = 149_597_870.7

def compute_unit_vectors(ra_deg, dec_deg, dra_deg_per_day, ddec_deg_per_day):
        mu_sun = 1.32712440018e11  # [km^3/s^2] gm of the sun

        # converting from degrees to radians
        ra  = np.deg2rad(ra_deg) 
        dec = np.deg2rad(dec_deg)
        l_hat, e_ra, e_dec = Sky_basis(ra, dec)
         # angular rate conversion from deg/day to rad/s
        dra  = np.deg2rad(dra_deg_per_day)  / 86400.0
        ddec = np.deg2rad(ddec_deg_per_day) / 86400.0

        v_hat = (dra * e_ra + ddec * e_dec)

        return l_hat, v_hat

def observables_to_geocentric_state(l_hat, v_hat, d_au, ddot_kms):   
        d_km = d_au * AU_km

        # multiplying l_hat by the rand dist. value gives actual geocentric pos vector of obj
        r_geo = np.outer(d_km, l_hat)

        # speed
        v_rad = np.outer(ddot_kms, l_hat)
        #v_tan = np.outer(d_km, v_hat)
        v_tan = d_km * v_hat
        v_geo = v_rad + v_tan
        
        return r_geo, v_geo

In [25]:
%%timeit
observables_to_geocentric_state(10, 20, 0.12, 0.22, 1., 4)

10.8 µs ± 34.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [77]:
l_hat, v_hat = compute_unit_vectors(10, 20, 0.12, 0.22)

In [38]:
%%timeit
observables_to_geocentric_state(l_hat, v_hat, 1., 4)

2.03 µs ± 3.41 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [70]:
d_au = np.arange(0, 2, 0.1)
ddot_kms = np.arange(3, 5, 0.1)

In [71]:
d_grid, ddot_grid = np.meshgrid(d_au, ddot_kms)
d_grid, ddot_grid = d_grid.flatten(), ddot_grid.flatten()

In [78]:
%%timeit
observables_to_geocentric_state(l_hat, v_hat, d_grid, ddot_grid)

10.5 µs ± 2.31 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [73]:
len(d_grid)

400

In [74]:
r_geo, v_geo = observables_to_geocentric_state(l_hat, v_hat, d_grid, ddot_grid)

In [75]:
r_geo.shape

(400, 3)

In [81]:
d_grid[10], ddot_grid[10], r_geo[10], v_geo[10]

(1.0,
 3.0,
 array([1.38440350e+08, 2.44107689e+07, 5.11654852e+07]),
 array([-0.09277346,  3.66593862,  7.27343484]))

In [5]:
def get_earth_and_observer(obstime_str):
    obstime = Time(obstime_str)
    
    rubin_location = EarthLocation.from_geodetic( 
        lon=-70.7366*u.deg,   # longitude (west is negative)
        lat=-30.2407*u.deg,   # latitude
        height=2647*u.m       # elevation
        
    )
    obs_geo = rubin_location.get_gcrs(obstime)
    r_obs = np.array([obs_geo.cartesian.x.to(u.km).value,
                      obs_geo.cartesian.y.to(u.km).value,
                      obs_geo.cartesian.z.to(u.km).value])
    
    # earth's barycentric vector rel to sun aka center of solsys to earth
    with solar_system_ephemeris.set('de432s'): 
        rE_bary, vE_bary = get_body_barycentric_posvel('earth', obstime)

    # convert to km and km/s arrays in cartesian coordinate system
    rE = np.array([rE_bary.x.to(u.km).value, #rE goes from sun to earth center
                   rE_bary.y.to(u.km).value,
                   rE_bary.z.to(u.km).value])
    
    vE = np.array([vE_bary.x.to(u.km/u.s).value, #vE goes from sun to earth center
                   vE_bary.y.to(u.km/u.s).value,
                   vE_bary.z.to(u.km/u.s).value])
    return obstime, rE, vE, r_obs

In [6]:
%%timeit
get_earth_and_observer("2025-09-16T00:00:00")

1.72 ms ± 318 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
rubin_location = EarthLocation.from_geodetic( 
        lon=-70.7366*u.deg,   # longitude (west is negative)
        lat=-30.2407*u.deg,   # latitude
        height=2647*u.m       # elevation
        
    )

        

def get_earth_and_observer(obstime_str):
    obstime = Time(obstime_str)
    
    obs_geo = rubin_location.get_gcrs(obstime)
    r_obs =  obs_geo.cartesian.xyz.to_value(u.km).T
    
    # earth's barycentric vector rel to sun aka center of solsys to earth
    with solar_system_ephemeris.set('de432s'): 
        rE_bary, vE_bary = get_body_barycentric_posvel('earth', obstime)

    # convert to km and km/s arrays in cartesian coordinate system
    rE  = rE_bary.xyz.to_value(u.km).T #rE goes from sun to earth center
    vE  = vE_bary.xyz.to_value(u.km/u.s).T

      #vE goes from sun to earth center

    return obstime, rE, vE, r_obs

In [8]:
%%timeit
get_earth_and_observer("2025-09-16T00:00:00")

1.27 ms ± 4.11 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
obstime, rE, vE, r_obs = get_earth_and_observer("2025-09-16T00:00:00")


In [10]:
rE

array([ 1.48729971e+08, -1.74898306e+07, -7.55902288e+06])

In [11]:
def sky_basis(ra_rad, dec_rad):
        l_hat = np.array([np.cos(dec_rad)*np.cos(ra_rad), # vector that points to object's pos on sky from observer
                          np.cos(dec_rad)*np.sin(ra_rad),
                          np.sin(dec_rad)])
        e_ra  = np.array([-np.sin(ra_rad), np.cos(ra_rad), 0.0]) # vector towards increasing RA - points east, 90 deg to l_hat
        e_dec = np.array([-np.sin(dec_rad)*np.cos(ra_rad), # vector towards increasing dec - points north
                          -np.sin(dec_rad)*np.sin(ra_rad),
                          np.cos(dec_rad)])
        return l_hat, e_ra, e_dec # turning angular rates into linear velocities


In [12]:
%%timeit
sky_basis(ra_rad = 0.1745, dec_rad = 0.3491)

5.38 μs ± 5.49 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [19]:
ra_deg, dec_deg, dra_deg_per_day, ddec_deg_per_day = 10.0, 20.0, 0.12, 0.22

In [20]:
def radec_rates_to_ecliptic_rates(ra_deg, dec_deg, dra_deg_day, ddec_deg_day):
    """
    Convert equatorial sky motion to ecliptic sky motion at the same direction.

    Parameters
    ----------
    ra_deg, dec_deg : float
        Right ascension and declination of the direction (deg).
    dra_deg_day, ddec_deg_day : float
        Angular rates on the sky (deg/day). dra is d(alpha)*cos(delta)/day.

    Returns
    -------
    v_lambda, v_beta : float
        Ecliptic longitude and latitude rates (deg/day).
    """
    ra  = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    
    e_ra  = np.array([-np.sin(ra),  np.cos(ra), 0.0])
    e_dec = np.array([-np.sin(dec)*np.cos(ra), -np.sin(dec)*np.sin(ra), np.cos(dec)])

    # Angular velocity vector in equatorial Cartesian 
    dra  = np.deg2rad(dra_deg_day)  / 86400.0
    ddec = np.deg2rad(ddec_deg_day) / 86400.0
    w_eq = dra*e_ra + ddec*e_dec

    # Rotation from equatorial to ecliptic 
    eps = np.deg2rad(23.439291)
    R = np.array([[ 1,           0,            0],
                  [ 0,  np.cos(eps),  np.sin(eps)],
                  [ 0, -np.sin(eps),  np.cos(eps)]])
    w_ecl = R @ w_eq


    
    l_hat_eq = np.array([np.cos(dec)*np.cos(ra),
                         np.cos(dec)*np.sin(ra),
                         np.sin(dec)])
    l_hat_ecl = R @ l_hat_eq
    lam = np.arctan2(l_hat_ecl[1], l_hat_ecl[0])
    bet = np.arcsin(l_hat_ecl[2])

    e_lam = np.array([-np.sin(lam),  np.cos(lam),  0.0])
    e_bet = np.array([-np.sin(bet)*np.cos(lam),
                      -np.sin(bet)*np.sin(lam),
                       np.cos(bet)])

    vlam  = np.dot(w_ecl, e_lam) * 86400.0 * 180.0/np.pi
    vbeta = np.dot(w_ecl, e_bet) * 86400.0 * 180.0/np.pi
    return vlam, vbeta

In [24]:
from astropy.coordinates import SkyCoord, GeocentricTrueEcliptic
import astropy.units as u

def radec_rates_to_ecliptic_rates1(ra_deg, dec_deg, dra_deg_day, ddec_deg_day):

    
    sc_eq = SkyCoord(
        ra=ra_deg * u.deg,
        dec=dec_deg * u.deg,
        pm_ra_cosdec=dra_deg_day * u.deg / u.day,
        pm_dec=ddec_deg_day * u.deg / u.day,
        frame="icrs"
    )

    
    sc_ecl = sc_eq.transform_to(GeocentricTrueEcliptic())

    
    vlam  = sc_ecl.pm_lon_coslat.to(u.deg / u.day).value
    vbeta = sc_ecl.pm_lat.to(u.deg / u.day).value

    return vlam, vbeta


In [22]:
%%timeit
radec_rates_to_ecliptic_rates(10.0, 20.0, 0.12, 0.22)

The slowest run took 6.45 times longer than the fastest. This could mean that an intermediate result is being cached.
34 ms ± 18.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [28]:
%%timeit
radec_rates_to_ecliptic_rates1(10.0, 20.0, 0.12, 0.22)

6.13 ms ± 6.08 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [29]:
%%timeit
radec_rates_to_ecliptic_rates(10.0, 20.0, 0.12, 0.22)

6.13 ms ± 9.69 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
3